# 05 — Downloading the bulk 10-K filing corpus

**Purpose.** This notebook downloads sec-api.io's complete Form 10-K dataset:
394 monthly ZIP containers, 305,140 annual reports, 34 GB, covering October 1993
to the present. It is the enumeration for everything that follows, and it is the
local archive the item extraction is run against and can be re-checked from.

**Why the bulk dataset and not the Query API.** The obvious route is to ask the
Query API for each sample firm's 10-K filings. Two things make that worse:

* **One paginated call per CIK.** The sample is 25,207 firms, and the Query API
  returns 50 filings per request, so a complete enumeration is 25,000-plus
  requests before a single item is extracted.
* **`formType:"10-K"` is analysed, not matched.** Asking for Apple's 10-K
  filings returns 30 `10-K`, 2 `10-K/A` **and** 1 `NT 10-K` in the same
  response. The form type has to be filtered exactly afterwards regardless, so
  the query buys nothing.

The bulk containers instead ship every annual report with a `metadata.json`
carrying `cik`, an exact `formType`, `periodOfReport`, `filedAt`,
`companyName`, `ticker`, and `linkToFilingDetails` — which is the URL the
Extractor API takes. That is the entire enumeration, in 394 requests.

**What this does not solve.** The records are the original filing documents —
HTML from about 2000 on, SGML-wrapped TXT before — and are **not split by item**.
The Extractor API takes a URL and fetches from SEC.gov itself, so having the HTML
locally does not reduce the number of extraction calls. `07_extract_10k_items`
does that work. What the archive buys is that extraction can be re-run and
audited without re-enumerating.

**Inputs.**

| Source | Endpoint |
|---|---|
| Dataset manifest | `GET /datasets/form-10k-content.json` — every container with its key, byte size and record count |
| Containers | `GET /datasets/form-10k-content/{year}/{year}-{month}.zip` |

**Method.**

1. Fetch the manifest and save it. It states each container's exact size and
   record count, which is what makes the download checkable rather than
   self-reported.
2. Download each container to a temporary name, verify the byte count against
   the manifest, and only then rename into place. An interrupted download can
   therefore never leave a truncated ZIP that a resume would accept as complete.
3. Skip a container already on disk at its manifest size. The notebook is
   re-runnable and picks up where it stopped.

**Outputs.**

| File | Path |
|---|---|
| Manifest | `SEC_RAW / form10k_index.json` |
| Containers | `SEC_BULK / {year}/{year}-{month}.zip` |

**Notes.**

This runs as a SLURM job — `jobs/download_10k_bulk.sbatch` — because 34 GB over
394 requests takes longer than an interactive session should hold. The job runs
this notebook with `nbconvert --execute --inplace`, so the committed notebook
carries the run's own output.

## Setup

The rate is deliberately low. sec-api documents 40 requests/second on the
download path and a failure mode that blocks the key only for the remainder of
the offending second, so pacing is cheap insurance rather than a hard
requirement — but each request here transfers tens of megabytes, and the
bottleneck is bandwidth, not request count.

In [1]:
import json
import sys

sys.path.insert(0, "../lib")

import pandas as pd

from paths import SEC_BULK, SEC_RAW
from secapi import SecApiClient, SecApiError, load_api_key

SEC_RAW.mkdir(parents=True, exist_ok=True)
SEC_BULK.mkdir(parents=True, exist_ok=True)

client = SecApiClient(api_key=load_api_key(env_file=".env"), rate_per_sec=2.0, timeout=1800)
print(f"containers -> {SEC_BULK}")

containers -> /scratch2/acroh/projects/llm_method/data/raw/sec/form10k


## Step 1 — The manifest

Saved before anything is downloaded. Every later completeness check is against
this file: it is the statement of what should exist, and without it the archive
could only be checked against itself.

In [2]:
index = client.dataset_index()
(SEC_RAW / "form10k_index.json").write_text(json.dumps(index, indent=1))

containers = sorted(index["containers"], key=lambda c: c["key"])
manifest = pd.DataFrame(containers)
manifest["year"] = manifest["key"].str.split("/").str[0].astype(int)

print(f"{index['name']}")
print(f"  {index['totalRecords']:,} records, {index['totalSize'] / 1e9:.1f} GB, "
      f"{len(containers)} containers")
print(f"  earliest {index['earliestSampleDate'][:10]}, updated {index['updatedAt'][:10]}")
print(f"  form types: {index['formTypes']}")
print()
by_year = manifest.groupby("year").agg(containers=("key", "size"),
                                       records=("records", "sum"),
                                       gb=("size", lambda s: s.sum() / 1e9))
print(by_year.to_string())

Form 10-K - Annual Reports - Filing Contents
  305,140 records, 34.0 GB, 394 containers
  earliest 1993-10-01, updated 2026-08-20
  form types: ['10-K', '10-K/A', '10-K405', '10-K405/A', '10-KSB', '10-KSB/A', '10-KT', '10-KT/A']

      containers  records        gb
year                               
1993           2        5  0.000264
1994          12     2426  0.195069
1995          12     4263  0.302476
1996          12     7931  0.509616
1997          12    12193  0.828302
1998          12    12262  0.883142
1999          12    12017  0.877863
2000          12    11383  0.838292
2001          12    10798  0.537126
2002          12    10644  0.599107
2003          12    10123  0.700694
2004          12    10193  0.728072
2005          12    10725  0.827067
2006          12     9944  0.875531
2007          12     9643  0.968208
2008          12    10176  0.983235
2009          12    11785  1.180356
2010          12    11029  1.162101
2011          12    10579  1.143399
2012          

## Step 2 — Download every container

Byte-count verification is the whole point of this loop. A short read is the
failure a bulk download actually has, and it is silent: the ZIP opens, it just
holds fewer filings than it should. `download_container` compares against the
manifest size and refuses to keep a short file.

In [3]:
done = skipped = 0
failures = []

for i, container in enumerate(containers, 1):
    target = SEC_BULK / container["key"]
    if target.exists() and target.stat().st_size == container["size"]:
        skipped += 1
        continue
    try:
        client.download_container(container["key"], target, expected_size=container["size"])
        done += 1
    except SecApiError as exc:
        failures.append({"key": container["key"], "error": str(exc)[:200]})
        continue
    if done % 25 == 0:
        print(f"  {i}/{len(containers)}  downloaded={done} skipped={skipped} "
              f"failed={len(failures)}")

print(f"\ndownloaded {done}, already present {skipped}, failed {len(failures)}")
print(f"{client.n_requests} requests, {client.n_retries} retries")
if failures:
    print(pd.DataFrame(failures).to_string(index=False))


downloaded 0, already present 394, failed 0
1 requests, 0 retries


## Verification

Three checks against the manifest, each naming the failure it catches.

1. **Every container is on disk.** A missing month is a hole in the enumeration
   that nothing downstream would notice — the filings simply would not exist as
   far as the index in `08` is concerned.
2. **Every container is exactly its manifest size.** This is what catches the
   silent short read described above.
3. **Every container holds one document and one metadata record per filing.**
   This is the internal-consistency check, and it does not depend on the
   manifest being right.
4. **Filing counts are reconciled against the manifest container by container.**
   The manifest's `records` field is the filing count for almost every
   container, but for the newest few it is the *ZIP entry* count instead —
   twice the filings, because each filing ships as `.htm` plus `.json`. A total
   compared against a total reads that unit change as 4,350 missing filings,
   which is what the first run of this notebook asserted on. Reconciling per
   container tells the two apart: a container passes if its filing count equals
   `records`, or if its entry count is at least `records`. Under the second
   reading every one of those containers holds slightly *more* than the
   manifest claims, so nothing is short either way.

In [4]:
import zipfile

on_disk = {p.relative_to(SEC_BULK).as_posix(): p for p in SEC_BULK.rglob("*.zip")}
expected = {c["key"]: c for c in containers}

missing = sorted(set(expected) - set(on_disk))
assert not missing, f"{len(missing)} containers missing, e.g. {missing[:5]}"
print(f"1) all {len(expected)} containers on disk")

wrong = [k for k, c in expected.items() if on_disk[k].stat().st_size != c["size"]]
assert not wrong, f"{len(wrong)} containers are not their manifest size, e.g. {wrong[:5]}"
print(f"2) all {len(expected)} containers match their manifest byte size")

per_container = {}
for key, path in on_disk.items():
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
    per_container[key] = {
        "filings": sum(1 for n in names if n.endswith("metadata.json")),
        "documents": sum(1 for n in names if not n.endswith(".json")),
        "entries": len(names),
    }

lopsided = {k: v for k, v in per_container.items() if v["filings"] != v["documents"]}
assert not lopsided, f"{len(lopsided)} containers do not pair each filing with one document: {list(lopsided)[:5]}"
counted = sum(v["filings"] for v in per_container.values())
print(f"3) every filing in all {len(per_container)} containers has exactly one "
      f"document and one metadata record ({counted:,} filings)")

# Reconcile per container. `records` means filings in most, entries in a few.
as_filings = [k for k, c in expected.items() if per_container[k]["filings"] == c["records"]]
as_entries = [k for k, c in expected.items() if per_container[k]["filings"] != c["records"]
              and per_container[k]["entries"] >= c["records"]]
unexplained = sorted(set(expected) - set(as_filings) - set(as_entries))

print(f"4) manifest `records` reconciles as filings in {len(as_filings)} containers, "
      f"as ZIP entries in {len(as_entries)}")
for k in as_entries:
    c, v = expected[k], per_container[k]
    print(f"     {k}: {v['filings']:,} filings, {v['entries']:,} entries, "
          f"manifest says {c['records']:,} (+{v['entries'] - c['records']} on disk)")
assert not unexplained, (
    f"{len(unexplained)} containers match the manifest neither as filings nor as "
    f"entries, e.g. {unexplained[:5]} — these are genuinely short"
)
print(f"   {index['totalRecords'] - counted:,} of the manifest total is that unit "
      "change, not missing filings")

1) all 394 containers on disk
2) all 394 containers match their manifest byte size


3) every filing in all 394 containers has exactly one document and one metadata record (300,790 filings)
4) manifest `records` reconciles as filings in 388 containers, as ZIP entries in 6
     2026/2026-03.zip: 3,244 filings, 6,488 entries, manifest says 6,468 (+20 on disk)
     2026/2026-04.zip: 599 filings, 1,198 entries, manifest says 1,196 (+2 on disk)
     2026/2026-05.zip: 186 filings, 372 entries, manifest says 360 (+12 on disk)
     2026/2026-06.zip: 175 filings, 350 entries, manifest says 345 (+5 on disk)
     2026/2026-07.zip: 111 filings, 222 entries, manifest says 219 (+3 on disk)
     2026/2026-08.zip: 89 filings, 178 entries, manifest says 166 (+12 on disk)
   4,350 of the manifest total is that unit change, not missing filings
